# 验证获取的数据

本 Notebook 用于读取 `data_fetch_ccxtpro` 模块采集并保存在 Parquet 文件中的高频行情数据（Trade 和 Orderbook），并进行输出验证。

In [ ]:
import os
import pandas as pd
from glob import glob

# 设置数据根目录 (根据你的 fetch_config.yaml 确定)
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')

print(f"Data Directory: {DATA_DIR}")

## 1. 验证 Trade 逐笔交易数据

In [ ]:
trade_files = glob(os.path.join(DATA_DIR, 'trades', '**', '*.parquet'), recursive=True)

if not trade_files:
    print("未找到任何 Trade 数据文件！请确认采集程序正在运行并在生成数据。")
else:
    print(f"找到了 {len(trade_files)} 个 Trade 数据文件。读取最新文件：")
    latest_trade_file = sorted(trade_files, key=os.path.getmtime)[-1]
    print(f"读取文件: {latest_trade_file}")
    
    try:
        df_trade = pd.read_parquet(latest_trade_file)
        print(f"\n数据形状: {df_trade.shape}")
        display(df_trade.tail())
        
        print("\n--- 基础统计描述 ---")
        display(df_trade[['price', 'amount']].describe())
    except Exception as e:
        print(f"读取 Trade Parquet 文件失败: {e}")

## 2. 验证 Orderbook 订单簿深度数据

In [ ]:
ob_files = glob(os.path.join(DATA_DIR, 'orderbooks', '**', '*.parquet'), recursive=True)

if not ob_files:
    print("未找到任何 Orderbook 数据文件！请确认采集程序正在运行并在生成数据。")
else:
    print(f"找到了 {len(ob_files)} 个 Orderbook 数据文件。读取最新文件：")
    latest_ob_file = sorted(ob_files, key=os.path.getmtime)[-1]
    print(f"读取文件: {latest_ob_file}")
    
    try:
        df_ob = pd.read_parquet(latest_ob_file)
        print(f"\n数据形状: {df_ob.shape}")
        display(df_ob.tail())
        
        print("\n--- 抽取部分核心字段 (L1档位) ---")
        cols_to_show = ['local_ts', 'exchange_ts', 'bid_p_1', 'bid_q_1', 'ask_p_1', 'ask_q_1']
        display(df_ob[cols_to_show].tail())
    except Exception as e:
        print(f"读取 Orderbook Parquet 文件失败: {e}")

## 3. 手动体检：扫描并清理损坏的 Parquet 文件

由于早前版本在使用 `fastparquet` 引擎配合 `append=True` 原地追加时，破坏了原文件的尾部元数据（Footer metadata），这会导致后续读取时报 `Parquet magic bytes not found` 或 `Invalid column metadata` 错误。

运行下方的格子可以遍历扫描所有 `.parquet` 文件，手动找出那些读取失败的数据（可选将其**直接删除**，以保证新进来的数据和分析流程干净）。

In [ ]:
def check_and_clean_corrupted_files(data_dir, delete_corrupted=False):
    print(f"🔍 开始扫描目录: {data_dir}")
    all_parquet_files = glob(os.path.join(data_dir, '**', '*.parquet'), recursive=True)
    
    corrupted_files = []
    for file_path in all_parquet_files:
        try:
            # 首选使用 pyarrow 引擎测试读取，如果不报错说明健康
            pd.read_parquet(file_path, engine='pyarrow')
        except Exception as e:
            corrupted_files.append((file_path, str(e)))
            
    print(f"\n扫描完成！共扫描 {len(all_parquet_files)} 个文件。")
    
    if corrupted_files:
        print(f"⚠️ 发现 {len(corrupted_files)} 个损坏的 Parquet 文件：")
        for path, err in corrupted_files:
            print(f" - {path}\n   └─ 报错信息: {err[:150]}...")
            
            if delete_corrupted:
                os.remove(path)
                print("     ✅ 已删除该损坏文件")
    else:
        print("✅ 未发现任何损坏文件，所有 Parquet 均可正常读取！")

# 默认只检查不删除，如果确认想一次性清理历史损坏文件，将下方的 False 改为 True
check_and_clean_corrupted_files(DATA_DIR, delete_corrupted=False)

## 4. 读取指定 Parquet 文件进行分析

这里演示如何直接读取具体的 Parquet 文件进行定制化的验证或分析。

In [ ]:
import pandas as pd
import os

# 你可以按需修改交易所、交易对、日期及具体的文件名 (这里以 Binance BTC/USDT spot 为例)
specific_ob_file = os.path.join(DATA_DIR, 'orderbooks', 'market_type=spot', 'exchange=binance', 'symbol=BTC_USDT', 'date=2026-02-24', '16.parquet')

if os.path.exists(specific_ob_file):
    print(f"正在读取指定的 Orderbook 文件: {specific_ob_file}")
    df_specific_ob = pd.read_parquet(specific_ob_file)
    print(f"Orderbook 数据形状: {df_specific_ob.shape}")
    display(df_specific_ob.head())
    
    # 计算并输出 Orderbook 数据的时间跨度
    if 'local_ts' in df_specific_ob.columns:
        min_ts = pd.to_datetime(df_specific_ob['local_ts'].min(), unit='ms')
        max_ts = pd.to_datetime(df_specific_ob['local_ts'].max(), unit='ms')
        duration = max_ts - min_ts
        print(f"\n⏰ Orderbook 数据时间跨度:")
        print(f"  开始时间: {min_ts}")
        print(f"  结束时间: {max_ts}")
        print(f"  总时段: {duration}")
else:
    print(f"找不到 Orderbook 文件: {specific_ob_file}")

# 验证 Trade 文件
specific_trade_file = os.path.join(DATA_DIR, 'trades', 'market_type=spot', 'exchange=binance', 'symbol=BTC_USDT', 'date=2026-02-24', '16.parquet')
if os.path.exists(specific_trade_file):
    print(f"\n{'-'*40}\n正在读取指定的 Trade 文件: {specific_trade_file}")
    df_specific_trade = pd.read_parquet(specific_trade_file)
    print(f"Trade 数据形状: {df_specific_trade.shape}")
    display(df_specific_trade.head())
    
    # 计算并输出 Trade 数据的时间跨度
    if 'local_ts' in df_specific_trade.columns:
        min_ts = pd.to_datetime(df_specific_trade['local_ts'].min(), unit='ms')
        max_ts = pd.to_datetime(df_specific_trade['local_ts'].max(), unit='ms')
        duration = max_ts - min_ts
        print(f"\n⏰ Trade 数据时间跨度:")
        print(f"  开始时间: {min_ts}")
        print(f"  结束时间: {max_ts}")
        print(f"  总时段: {duration}")
else:
    print(f"\n找不到 Trade 文件: {specific_trade_file}")


## 5. 分钟级文件校验 (完整性检查)

由于采集策略已更新为按分钟保存 (`%H%M.parquet`)，本功能用于检查特定交易所、交易对在某一日期的特定小时内，是否存在数据缺失或损坏文件隔离区 (`.corrupted_xxx`) 的遗留。

In [ ]:
import os
import pandas as pd
from glob import glob
from collections import defaultdict

target_date = '2026-02-24'
target_hour = '18' # 请修改为您想要检查的小时段

def validate_all_hourly_completeness(data_dir, date_str, hour_str):
    print(f"\n🔍 开始全局扫描 {date_str} [{hour_str}:00 ~ {hour_str}:59] 时段的所有数据...\n")
    
    # 收集所有的 market_type 路径下所有的 exchange / symbol
    search_pattern = os.path.join(data_dir, '*', 'market_type=*', 'exchange=*', 'symbol=*', f'date={date_str}')
    date_folders = glob(search_pattern)
    
    results = []
    
    for folder in sorted(date_folders):
        # 解析路径结构来获取 data_type, market_type, exchange, symbol
        parts = folder.split(os.sep)
        try:
            data_type = parts[parts.index('raw') + 1]
            market_type = parts[-4].split('=')[1]
            exchange = parts[-3].split('=')[1]
            symbol = parts[-2].split('=')[1]
        except:
            continue # 如果目录结构异常则跳过
            
        # 查找该小时的所有正常 Parquet 文件 (如 1700.parquet 到 1759.parquet)
        pattern = os.path.join(folder, f'{hour_str}[0-9][0-9].parquet')
        normal_files = glob(pattern)
        
        # 过滤掉体积为 0 或因无数据极小的空 Parquet (比如只有几百字节的那种 metadata 壳子)
        valid_files = [f for f in normal_files if os.path.getsize(f) > 1024]
        
        # 查找该小时的隔离损坏文件
        corrupted_pattern = os.path.join(folder, f'{hour_str}[0-9][0-9].parquet.corrupted_*')
        corrupted_files = glob(corrupted_pattern)
        
        # 真正的 Empty 包括两件事：
        # 1. 有这分钟的文件，但是空壳子 (len(normal) - len(valid))
        # 2. 彻底连这分钟的文件都没有产生 (60 - len(normal_files) - len(corrupted_files))
        corrupted_count = len(corrupted_files)
        valid_count = len(valid_files)
        
        # 如果整个小时都没有产生过一丝文件，那就彻底跳过不展示在表里（说明还没运行或者关着）
        if len(normal_files) == 0 and corrupted_count == 0:
            continue
            
        # 如果由于某种原因这一小时超过了 60 个文件（例如隔离了又被重写），取最大值防护负数计算
        total_files_found = len(normal_files) + corrupted_count
        expected_total = 60
        # 实际的缺失/空文件数：总计 60 减去有内容的，减去损坏保留了的
        empty_and_missing_count = expected_total - valid_count - corrupted_count
        if empty_and_missing_count < 0:
            empty_and_missing_count = 0
            
        # 尝试提取时间跨度
        time_span = "N/A"
        if valid_files:
            valid_files.sort()
            try:
                df_first = pd.read_parquet(valid_files[0])
                df_last = pd.read_parquet(valid_files[-1])
                
                ts_col_first = 'local_ts' if 'local_ts' in df_first.columns else ('timestamp' if 'timestamp' in df_first.columns else None)
                ts_col_last = 'local_ts' if 'local_ts' in df_last.columns else ('timestamp' if 'timestamp' in df_last.columns else None)
                
                if ts_col_first and ts_col_last:
                    min_ts = pd.to_datetime(df_first[ts_col_first].min(), unit='ms')
                    max_ts = pd.to_datetime(df_last[ts_col_last].max(), unit='ms')
                    time_span = f"{min_ts.strftime('%M:%S')} ~ {max_ts.strftime('%M:%S')}"
            except Exception as e:
                time_span = f"读取错误: {str(e)[:20]}"
        
        results.append({
            'Data': data_type.capitalize(),
            'Type': market_type,
            'Exchange': exchange,
            'Symbol': symbol,
            'Normal': valid_count,
            'Empty': empty_and_missing_count,
            'Corrupted': corrupted_count,
            'Time Span': time_span
        })
        
    if not results:
        print(f"❌ 在 {date_str} 的 {hour_str} 小时段内，未找到任何交易所的任何数据。")
        return
        
    # 格式化输出为 Pandas DataFrame
    df_report = pd.DataFrame(results)
    print("📊 各交易所及币种在该时间段的数据收集状态报告:")
    print("="*85)
    display(df_report)
    print("="*85)
    print("💡 说明:")
    print("- Normal (正常): 大于 1KB，有实际内容的数据块文件。")
    print("- Empty (空文件/无数据): 该分钟内没有发生交易导致连文件都未产生，或虽然产生了但内容为空（计算逻辑：60 - Normal - Corrupted）。")
    print("- Corrupted (损坏): 发生写入阻断，被程序隔离起来抢救的 '.corrupted_xxx' 文件。")
    print("- 以上三者相加在绝大多数情况下将完美恒等于一小时的 60 分钟。")

validate_all_hourly_completeness(DATA_DIR, target_date, target_hour)


## 6. 合并分钟级碎文件为小时/日级别文件

为了方便后续建模或回测使用，将细碎的分钟级文件 (`1700.parquet`, `1701.parquet`...) 合并回标准的小时级文件 (`17.parquet`) 或全天文件 (`2026-02-24.parquet`)。

In [ ]:
import os
import pandas as pd
from glob import glob

def compact_parquet_files(data_dir, data_type, exchange, symbol, date_str, target_hour=None, delete_originals=False):
    """
    target_hour: 如果传入如 '17'，则合并 17:00 ~ 17:59 的文件为 17.parquet。
                 如果为 None，则尝试合并该目录下所有的文件为 consolidated.parquet。
    """
    base_path = os.path.join(data_dir, data_type, 'market_type=spot', f'exchange={exchange}', f'symbol={symbol}', f'date={date_str}')
    if not os.path.exists(base_path):
        print(f"目录不存在，无法合并: {base_path}")
        return

    if target_hour:
        # 匹配 1700.parquet ~ 1759.parquet (由于是4位数)
        pattern = os.path.join(base_path, f'{target_hour}[0-9][0-9].parquet')
        output_file = os.path.join(base_path, f'{target_hour}.parquet')
    else:
        # 匹配全部
        pattern = os.path.join(base_path, '*.parquet')
        output_file = os.path.join(base_path, 'consolidated_all.parquet')
        
    files_to_merge = sorted(glob(pattern))
    
    # 排除输出文件自身（防止重复合并）
    files_to_merge = [f for f in files_to_merge if os.path.basename(f) not in [f'{target_hour}.parquet', 'consolidated_all.parquet']]

    if not files_to_merge:
        print(f"未找到符合条件的分钟碎片文件。Pattern: {pattern}")
        return
        
    print(f"🔄 准备合并 {len(files_to_merge)} 个小文件 -> {output_file} ...")
    
    dfs = []
    valid_files = []
    for f in files_to_merge:
        try:
            df = pd.read_parquet(f)
            dfs.append(df)
            valid_files.append(f)
        except Exception as e:
            print(f"⚠️ 跳过无法读取的文件 {f}: {e}")
            
    if not dfs:
        print("没有有效的文件可合并。")
        return
        
    # 实施合并
    merged_df = pd.concat(dfs, ignore_index=True)
    
    # 按照时间戳排序，保证数据严谨性
    if 'local_ts' in merged_df.columns:
        merged_df = merged_df.sort_values('local_ts').reset_index(drop=True)
        
    # 写入新的大文件
    merged_df.to_parquet(output_file, engine='pyarrow', compression='snappy')
    print(f"✅ 合并成功！新文件形状: {merged_df.shape}, 路径: {output_file}")
    
    # 删除清理机制
    if delete_originals:
        print(f"🗑️ 正在清理 {len(valid_files)} 个原始分钟碎片文件...")
        for f in valid_files:
            os.remove(f)
        print("清理完成。")

# 示例运行: 合并 Trade 目录下 17:00 阶段的数据
# 将 delete_originals 改为 True 即可在合并成功后删除原有的碎块文件以节省空间
compact_parquet_files(DATA_DIR, 'trades', target_exchange, target_symbol, target_date, target_hour='17', delete_originals=False)
